In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path.cwd().parent))

from data.loader import load_tables, load_excel_files
from data.calculations import (
    build_product_classification,
    build_product_families,
    enrich_sales_with_classification,
    build_customer_activity,
    enrich_customers_with_activity,
    enrich_sales_with_customer_activity,
    filter_sales_by_bodega,
    prepare_sales
)

from data.google_sheets import load_visitas_from_google_sheet

# -----------------------------
# Load raw data
# -----------------------------
access_file = "/Users/ricardolugo/Library/CloudStorage/OneDrive-Personal/LH/Reports/sales_lh.accdb"
tables = ["sales", "customers"]

data = load_tables(access_file, tables)

df_sales = data["sales"]
df_customers = data["customers"]
df_actividades, df_clasificacion, df_inventario, df_crm, df_cotizacion = load_excel_files()

df_visitas = load_visitas_from_google_sheet()

# -----------------------------
# Build reference tables
# -----------------------------
df_grupos = build_product_classification(df_clasificacion)
df_familias = build_product_families(df_clasificacion)
df_customer_activity = build_customer_activity(df_actividades)

# -----------------------------
# Enrich tables
# -----------------------------
df_sales_enriched = enrich_sales_with_classification(
    df_sales,
    df_grupos,
    df_familias
)

df_customers_enriched = enrich_customers_with_activity(
    df_customers,
    df_customer_activity
)

df_sales_final = enrich_sales_with_customer_activity(
    df_sales_enriched,
    df_customers_enriched
)

# -----------------------------
# Working filter: Cali
# -----------------------------

df_sales_clean = prepare_sales(df_sales_final)

df_cali = filter_sales_by_bodega(df_sales_clean, 50)
df_bogota = filter_sales_by_bodega(df_sales_clean, 1)

Loading sales...
sales loaded: (285619, 25)
Loading customers...
customers loaded: (36429, 16)
Actividades shape: (594, 6)
Clasificacion shape: (203, 4)
Inventario shape: (11169, 32)
CRM shape: (539, 16)
Cotizacion shape: (374, 12)


In [11]:
df_cali["fecha"] = pd.to_datetime(df_cali["fecha"], errors="coerce")

today = df_cali["fecha"].max()
start_date = today - pd.DateOffset(months=12)

df_last12 = df_cali[df_cali["fecha"] >= start_date].copy()

# MUY IMPORTANTE
df_last12["valorbruto"] = pd.to_numeric(
    df_last12["valorbruto"],
    errors="coerce"
).fillna(0)

top_clientes = (
    df_last12
    .groupby(
        ["nit", "razonsocial", "actividad_economica"],
        as_index=False
    )
    .agg(
        ventas_12m=("valorbruto", "sum"),
        compras=("numero", "nunique"),
        ultima_compra=("fecha", "max")
    )
)

top_clientes["dias_sin_comprar"] = (
    today - top_clientes["ultima_compra"]
).dt.days

top_clientes = (
    top_clientes
    .sort_values("ventas_12m", ascending=False)
    .head(20)
)

top_clientes["ventas_12m_mm"] = (
    top_clientes["ventas_12m"] / 1_000_000
).round(1)

top_clientes["ventas_12m_display"] = (
    "$" + top_clientes["ventas_12m_mm"].map("{:,.1f} MM".format)
)

actividades_excluir = [

    "COMERCIO EN GENERAL",

    "COMERCIO DE RODAMIENTOS Y AFINES",

    "COMERCIO Y MANTENIMIENTO DE AUTOMOTORES"

]

top_clientes = top_clientes[

    ~top_clientes["actividad_economica"].isin(actividades_excluir)

].copy()


top_clientes[
    [
        "razonsocial",
        "ventas_12m_display",
        "compras",
        "ultima_compra",
        "dias_sin_comprar",
        "actividad_economica"
    ]
]



,razonsocial,ventas_12m_display,compras,ultima_compra,dias_sin_comprar,actividad_economica
234,CARTONES AMERICA S.A.,$557.2 MM,309,2026-03-25,6,FABRICACION PRODUCTOS CARTON Y PAPEL
402,SERMOTOR INGENIERIA S.A.S,$215.7 MM,123,2026-03-31,0,TALLERES
290,IME INGENIERIA DE MAQUINAS ELECTRICAS S.A.S,$135.9 MM,105,2026-03-27,4,TALLERES
222,NESTLE DE COLOMBIA S.A.,$133.1 MM,43,2026-03-12,19,PRODUCCION Y TRANSF. ALIMENTICIOS Y BEBIDAS
311,PAPELES NACIONALES S.A.S,$131.7 MM,106,2026-03-20,11,FABRICACION PRODUCTOS CARTON Y PAPEL
246,CARTON DE COLOMBIA S.A.,$121.6 MM,50,2026-03-27,4,FABRICACION PULPA DE PAPEL
182,PGI COLOMBIA LTDA.,$118.5 MM,44,2026-03-19,12,PRODUCTOS TEXTILES
128,POLLOS EL BUCANERO S.A.,$106.6 MM,50,2026-03-17,14,PRODUCCION Y TRANSF. ALIMENTICIOS Y BEBIDAS
353,REFINADORA NACIONAL DE ACEITES Y GRASAS S.A.S,$103.9 MM,70,2026-03-25,6,PRODUCCION Y TRANSF. ALIMENTICIOS Y BEBIDAS
414,CENTRAL DE REBOBINADOS MOTORES ELECTRICOS S.A.S.,$103.7 MM,53,2026-03-20,11,TALLERES


In [12]:
# =========================
# LIMPIAR VISITAS
# =========================

import pandas as pd

df_visitas["Cliente_Dashboard"] = (
    df_visitas["Cliente_Nombre"]
    .astype("object")
    .replace("", pd.NA)
    .fillna(df_visitas["Cliente"].astype(str))
    .astype(str)
)

df_visitas["Cliente_Dashboard"] = (
    df_visitas["Cliente_Dashboard"]
    .str.upper()
    .str.strip()
)

df_visitas["Fecha_Visita"] = pd.to_datetime(
    df_visitas["Fecha_Visita"],
    errors="coerce"
)

df_visitas["Fecha_Compromiso"] = pd.to_datetime(
    df_visitas["Fecha_Compromiso"],
    errors="coerce"
)

df_visitas["Requiere_Accion"] = (
    df_visitas["Requiere_Accion"]
    .astype(str)
    .str.upper()
    .eq("TRUE")
)

# =========================
# SEMÁFORO CLIENTE
# =========================

def semaforo_cliente(dias):
    if dias <= 15:
        return "Verde"
    elif dias <= 30:
        return "Amarillo"
    else:
        return "Rojo"


top_clientes["semaforo"] = top_clientes["dias_sin_comprar"].apply(semaforo_cliente)

# =========================
# ACCIONES ABIERTAS
# =========================

acciones_pendientes_df = df_visitas[
    (df_visitas["Requiere_Accion"] == True) &
    (
        df_visitas["Estado"]
        .astype(str)
        .str.upper()
        .isin(["ABIERTO", "EN SEGUIMIENTO", "PENDIENTE"])
    )
].copy()

# =========================
# RESUMEN VISITAS
# =========================

resumen_visitas = (
    df_visitas
    .groupby("Cliente_Dashboard", as_index=False)
    .agg(
        ultima_visita=("Fecha_Visita", "max"),
        total_visitas=("ID_Visita", "count")
    )
)

# =========================
# RESUMEN ACCIONES
# =========================

resumen_acciones = (
    acciones_pendientes_df
    .groupby("Cliente_Dashboard", as_index=False)
    .agg(
        acciones_pendientes=("ID_Visita", "count"),
        proxima_fecha_compromiso=("Fecha_Compromiso", "min")
    )
)

# =========================
# MERGE FINAL
# =========================

dashboard_clientes = (
    top_clientes
    .merge(
        resumen_visitas,
        left_on="razonsocial",
        right_on="Cliente_Dashboard",
        how="left"
    )
    .merge(
        resumen_acciones,
        left_on="razonsocial",
        right_on="Cliente_Dashboard",
        how="left"
    )
)

# =========================
# DÍAS SIN VISITA
# =========================

today = pd.Timestamp.today().normalize()

dashboard_clientes["dias_sin_visita"] = (
    today - dashboard_clientes["ultima_visita"]
).dt.days

# =========================
# LIMPIEZA VISUAL
# =========================

dashboard_clientes["acciones_pendientes"] = (
    dashboard_clientes["acciones_pendientes"]
    .fillna(0)
    .astype(int)
)

dashboard_clientes["ultima_visita_display"] = (
    dashboard_clientes["ultima_visita"]
    .dt.strftime("%Y-%m-%d")
)

dashboard_clientes["ultima_visita_display"] = (
    dashboard_clientes["ultima_visita_display"]
    .fillna("No visitado")
)

dashboard_clientes["proxima_fecha_compromiso_display"] = (
    dashboard_clientes["proxima_fecha_compromiso"]
    .dt.strftime("%Y-%m-%d")
)

dashboard_clientes["proxima_fecha_compromiso_display"] = (
    dashboard_clientes["proxima_fecha_compromiso_display"]
    .fillna("Sin pendiente")
)

dashboard_clientes["dias_sin_visita"] = (
    dashboard_clientes["dias_sin_visita"]
    .fillna(999)
    .astype(int)
)

# =========================
# TABLA FINAL
# =========================


tabla_top_clientes = dashboard_clientes[
    [
        "razonsocial",
        "actividad_economica",
        "ventas_12m",
        "ventas_12m_display",
        "semaforo",
        "ultima_visita_display",
        "dias_sin_visita",
        "acciones_pendientes",
        "proxima_fecha_compromiso_display",
    ]
].sort_values("ventas_12m", ascending=False)

tabla_top_clientes["dias_sin_visita_display"] = tabla_top_clientes["dias_sin_visita"].replace(999, "No visitado")

tabla_top_clientes

,razonsocial,actividad_economica,ventas_12m,ventas_12m_display,semaforo,ultima_visita_display,dias_sin_visita,acciones_pendientes,proxima_fecha_compromiso_display,dias_sin_visita_display
0,CARTONES AMERICA S.A.,FABRICACION PRODUCTOS CARTON Y PAPEL,5.571873e+08,$557.2 MM,Verde,2026-04-25,4,0,Sin pendiente,4
1,SERMOTOR INGENIERIA S.A.S,TALLERES,2.157416e+08,$215.7 MM,Verde,No visitado,999,0,Sin pendiente,No visitado
2,IME INGENIERIA DE MAQUINAS ELECTRICAS S.A.S,TALLERES,1.359207e+08,$135.9 MM,Verde,No visitado,999,0,Sin pendiente,No visitado
3,NESTLE DE COLOMBIA S.A.,PRODUCCION Y TRANSF. ALIMENTICIOS Y BEBIDAS,1.330935e+08,$133.1 MM,Amarillo,No visitado,999,0,Sin pendiente,No visitado
4,PAPELES NACIONALES S.A.S,FABRICACION PRODUCTOS CARTON Y PAPEL,1.317378e+08,$131.7 MM,Verde,No visitado,999,0,Sin pendiente,No visitado
5,CARTON DE COLOMBIA S.A.,FABRICACION PULPA DE PAPEL,1.216071e+08,$121.6 MM,Verde,No visitado,999,0,Sin pendiente,No visitado
6,PGI COLOMBIA LTDA.,PRODUCTOS TEXTILES,1.185368e+08,$118.5 MM,Verde,No visitado,999,0,Sin pendiente,No visitado
7,POLLOS EL BUCANERO S.A.,PRODUCCION Y TRANSF. ALIMENTICIOS Y BEBIDAS,1.065697e+08,$106.6 MM,Verde,No visitado,999,0,Sin pendiente,No visitado
8,REFINADORA NACIONAL DE ACEITES Y GRASAS S.A.S,PRODUCCION Y TRANSF. ALIMENTICIOS Y BEBIDAS,1.038638e+08,$103.9 MM,Verde,No visitado,999,0,Sin pendiente,No visitado
9,CENTRAL DE REBOBINADOS MOTORES ELECTRICOS S.A.S.,TALLERES,1.037192e+08,$103.7 MM,Verde,No visitado,999,0,Sin pendiente,No visitado


In [14]:
# =========================
# TABLA OPERATIVA POR ASESOR
# =========================

import pandas as pd

df_visitas = df_visitas.copy()

# -------------------------
# Limpieza de fechas
# -------------------------
df_visitas["Fecha_Visita"] = pd.to_datetime(
    df_visitas["Fecha_Visita"],
    errors="coerce"
)

df_visitas["Fecha_Compromiso"] = pd.to_datetime(
    df_visitas["Fecha_Compromiso"],
    errors="coerce"
)

# -------------------------
# Normalizar booleanos
# -------------------------
df_visitas["Requiere_Accion"] = (
    df_visitas["Requiere_Accion"]
    .astype(str)
    .str.upper()
    .eq("TRUE")
)

df_visitas["Generar_Oportunidad_CRM"] = (
    df_visitas["Generar_Oportunidad_CRM"]
    .astype(str)
    .str.upper()
    .eq("TRUE")
)

# -------------------------
# Cliente limpio para dashboard
# -------------------------
cliente_nombre = (
    df_visitas["Cliente_Nombre"]
    .astype("object")
    .replace("", pd.NA)
)

cliente_fallback = (
    df_visitas["Cliente"]
    .astype("object")
    .replace("", pd.NA)
)

df_visitas["Cliente_Dashboard"] = (
    cliente_nombre
    .fillna(cliente_fallback)
    .fillna("Sin cliente")
    .astype(str)
    .str.upper()
    .str.strip()
)

# -------------------------
# Tabla operativa principal
# -------------------------
tabla_asesor = df_visitas[
    [
        "Asesor",
        "Cliente_Dashboard",
        "Fecha_Visita",
        "Tipo_Visita",
        "Requiere_Accion",
        "Accion_Requerida",
        "Fecha_Compromiso",
        "Estado",
        "Generar_Oportunidad_CRM",
        "Responsable_Seguimiento",
    ]
].copy()

# -------------------------
# Formato visual
# -------------------------
tabla_asesor["Fecha_Visita"] = (
    tabla_asesor["Fecha_Visita"]
    .dt.strftime("%Y-%m-%d")
    .fillna("Sin visita")
)

tabla_asesor["Fecha_Compromiso"] = (
    tabla_asesor["Fecha_Compromiso"]
    .dt.strftime("%Y-%m-%d")
    .fillna("Sin fecha")
)

tabla_asesor["Estado"] = tabla_asesor["Estado"].astype(str).fillna("")
tabla_asesor["Asesor"] = tabla_asesor["Asesor"].astype(str).fillna("")
tabla_asesor["Responsable_Seguimiento"] = (
    tabla_asesor["Responsable_Seguimiento"]
    .astype(str)
    .replace("nan", "")
)

# -------------------------
# Renombrar columnas
# -------------------------
tabla_asesor = tabla_asesor.rename(columns={
    "Cliente_Dashboard": "Cliente",
    "Fecha_Visita": "Fecha visita",
    "Tipo_Visita": "Tipo visita",
    "Requiere_Accion": "Requiere acción",
    "Accion_Requerida": "Acción requerida",
    "Fecha_Compromiso": "Fecha compromiso",
    "Generar_Oportunidad_CRM": "Crear oportunidad CRM",
    "Responsable_Seguimiento": "Responsable seguimiento",
})

# -------------------------
# Tabla de pendientes
# -------------------------
tabla_pendientes = tabla_asesor[
    (tabla_asesor["Requiere acción"] == True) &
    (tabla_asesor["Estado"].astype(str).str.upper() != "CERRADO")
].copy()

# -------------------------
# Orden
# -------------------------
tabla_pendientes = tabla_pendientes.sort_values(
    ["Asesor", "Fecha compromiso"],
    ascending=[True, True]
)

tabla_asesor = tabla_asesor.sort_values(
    ["Asesor", "Fecha compromiso"],
    ascending=[True, True]
)

tabla_asesor

,Asesor,Cliente,Fecha visita,Tipo visita,Requiere acción,Acción requerida,Fecha compromiso,Estado,Crear oportunidad CRM,Responsable seguimiento
0,Fabio,CARTONES,2026-04-28,Seguimiento,True,Analisis alternativas en marcas de bajo costo ...,2026-04-30,Abierto,True,Ricardo
1,Jairo,6,2026-04-27,Comercial,False,,Sin fecha,En seguimiento,False,
2,Jeisman,CARTONES AMERICA S.A.,2026-04-25,Postventa,False,,Sin fecha,Abierto,False,


In [15]:
tabla_asesor

,Asesor,Cliente,Fecha visita,Tipo visita,Requiere acción,Acción requerida,Fecha compromiso,Estado,Crear oportunidad CRM,Responsable seguimiento
0,Fabio,CARTONES,2026-04-28,Seguimiento,True,Analisis alternativas en marcas de bajo costo ...,2026-04-30,Abierto,True,Ricardo
1,Jairo,6,2026-04-27,Comercial,False,,Sin fecha,En seguimiento,False,
2,Jeisman,CARTONES AMERICA S.A.,2026-04-25,Postventa,False,,Sin fecha,Abierto,False,


In [32]:
clientes_appsheet = (
    df_cali[
        [
            "nit",
            "razonsocial",
            "actividad_economica"
        ]
    ]
    .drop_duplicates(subset=["nit"])
    .copy()
)

clientes_appsheet = clientes_appsheet.merge(
    df_customers_enriched[["nit", "vendedor"]],
    on="nit",
    how="left"
)

clientes_appsheet = clientes_appsheet.rename(columns={
    "nit": "ID_CLIENTE",
    "razonsocial": "Cliente",
    "vendedor": "Vendedor_Asignado",
    "actividad_economica": "Actividad_Economica"
})

clientes_appsheet = clientes_appsheet[
    [
        "ID_CLIENTE",
        "Cliente",
        "Vendedor_Asignado",
        "Actividad_Economica"
    ]
].sort_values("Cliente")

clientes_appsheet.head()

,ID_CLIENTE,Cliente,Vendedor_Asignado,Actividad_Economica
750,901685224,24/7 METALMECANICAS S.A.S.,ALMACEN CALI -UNO-,TALLERES
1139,900988807,3Z INGENIERIA EN PLASTICOS S.A.S.,ALMACEN CALI -UNO-,PRODUCTOS PLASTICOS
428,901778896,A&C RODAMIENTOS S.A.S.,JOHAN SEBASTIAN CORDOBA GAMBOA,COMERCIO DE RODAMIENTOS Y AFINES
1226,805005216,A.G REPUESTOS Y SERVICIOS S.A.S,ALMACEN CALI -UNO-,COMERCIO Y MANTENIMIENTO DE AUTOMOTORES
1415,94294075,ABADIA MUÑOZ JAVIER,NaN,NaN


In [33]:
clientes_appsheet = (
    clientes_appsheet
    .drop_duplicates(subset=["ID_CLIENTE"])
    .sort_values("Cliente")
    .reset_index(drop=True)
)

clientes_appsheet.to_excel(
    "clientes_appsheet.xlsx",
    index=False
)

print("Archivo exportado sin duplicados: clientes_appsheet.xlsx")

Archivo exportado sin duplicados: clientes_appsheet.xlsx


In [30]:
print(clientes_appsheet.columns.tolist())

['ID_CLIENTE', 'Cliente', 'Vendedor_Asignado', 'Actividad_Economica']
